# proSIM - SIM misuse detection: model training

Trains and evaluates two models on the synthetic CDR data (one row per SIM per day):

| Model | Type | Role |
|---|---|---|
| **Isolation Forest** | Unsupervised | Flags SIM-days with unusual usage, without using labels |
| **Random Forest** | Supervised (multiclass) | Classifies each SIM-day: `normal`, `unauthorized_sharing`, `data_reselling`, `ghost_usage`, `inactive_billing` |

Each section matches an issue in the **ML Training** milestone. Kernel: **proSIM (ml-service venv)**.

## 1. Setup and load data  (issue #1)

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

SEED = 42
np.random.seed(SEED)

# Locate the ml-training folder whether the kernel starts in notebooks/ or elsewhere.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and p.name == "ml-training")
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
MODELS_DIR = ROOT.parent / "ml-service" / "models"
REPORTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
print("Data:", DATA_DIR)

In [ ]:
agg = pd.read_csv(DATA_DIR / "synthetic_cdr_daily_agg.csv")
labels = pd.read_csv(DATA_DIR / "synthetic_cdr_daily_labels.csv")

df = agg.merge(labels, on=["sim_id", "date"], how="left", validate="one_to_one")
assert df["label"].notna().all(), "every SIM-day must have a label"
df = df.sort_values(["sim_id", "date"]).reset_index(drop=True)

print(f"{len(df):,} SIM-days | {df.sim_id.nunique():,} SIMs | {df.date.min()} to {df.date.max()}")
df.head()

In [ ]:
# Columns and types, to plan feature engineering
df.dtypes.to_frame("dtype").T

## 2. Exploratory data analysis  (issue #2)

- [ ] Label distribution table and chart (about 5% misuse)
- [ ] Compare usage patterns of each misuse type against normal
- [ ] Check and document that `billed_mb > total_mb` exactly separates inactive billing

## 3. Feature engineering  (issue #3)

- [ ] Raw daily usage features
- [ ] Usage-pattern ratios: upload share, off-peak share, hotspot share, public-APN share, MB per session
- [ ] Billing gap: `billed_mb - total_mb`, billed / used
- [ ] Per-SIM 14-day baseline using PAST days only (`shift(1)` before the rolling window): ratio to median, z-score
- [ ] One-hot encode `tier`; never use `label`, `sim_id` or `date` as features

## 4. Train/test split by SIM  (issue #4)

- [ ] `GroupShuffleSplit(test_size=0.25, random_state=SEED)` with `groups=df['sim_id']`
- [ ] Assert no SIM is in both sets; report the misuse share of each

## 5. Isolation Forest  (issue #5)

- [ ] Fit on training features WITHOUT labels; `contamination` = misuse share (~0.05); `n_jobs=-1`
- [ ] Anomaly score = `-score_samples(X)`; threshold = top `contamination` share of training scores
- [ ] Evaluate with labels: precision, recall, F1, ROC-AUC, PR-AUC, recall per misuse type

## 6. Random Forest  (issue #6)

- [ ] Multiclass, `class_weight='balanced_subsample'`, 300-400 trees, `n_jobs=-1`
- [ ] Optional: tune `max_depth` / `min_samples_leaf` with `GroupKFold` on the training SIMs
- [ ] Classification report per class + confusion matrix (save to `REPORTS_DIR`)

## 7. Compare models  (issue #7)

- [ ] Comparison table: Isolation Forest vs Random Forest
- [ ] Precision-recall curves and feature importance chart (save to `REPORTS_DIR`)
- [ ] Notes for the write-up: inactive-billing separability, data-reselling weakness

## 8. Save models and metadata  (issue #8)

- [ ] Refit both models on ALL SIMs; save with `joblib` to `MODELS_DIR`
- [ ] Save metadata JSON: feature list in order, Isolation Forest threshold, class names, sklearn version
- [ ] Save metrics JSON to `REPORTS_DIR`